In [1]:
!pip install scikit-surprise

In [2]:
import pandas as pd
import numpy as np
from collections import defaultdict

from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split



In [3]:
!pip install numpy==1.26.4

# Task
Uninstall `numpy` and `scikit-surprise`, then install `numpy==1.24.3`, and finally reinstall `scikit-surprise` to resolve compatibility issues.

## Clean and Reinstall Libraries

### Subtask:
Uninstall existing versions of `numpy` and `scikit-surprise` to ensure a clean slate, then install a `numpy` version (`numpy==1.24.3`) compatible with `scikit-surprise`'s internal C extensions, and finally reinstall `scikit-surprise` to link against the correct `numpy` version.


**Reasoning**:
The first instruction is to uninstall the current version of `numpy`. I will use `!pip uninstall numpy -y` to automate confirmation.



In [4]:
get_ipython().system('pip uninstall numpy -y')

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


**Reasoning**:
Following the instructions, the next step is to uninstall the current version of `scikit-surprise`.



In [5]:
get_ipython().system('pip uninstall scikit-surprise -y')

Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4


In [6]:
import pandas as pd
import numpy as np
from collections import defaultdict

from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [7]:
user_item_matrix = pd.read_csv("user_item_matrix (3).csv", index_col=0)

data_long = user_item_matrix.stack().reset_index()
data_long.columns = ['user_id', 'product_id', 'interaction']
data_long = data_long[data_long['interaction'] > 0]


In [8]:
reader = Reader(rating_scale=(1, 3))
data = Dataset.load_from_df(
    data_long[['user_id','product_id','interaction']], reader
)

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)


In [9]:
baseline_model = SVD(n_factors=50, n_epochs=20)
baseline_model.fit(trainset)


In [10]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=5, threshold=1.0):
    user_est_true = defaultdict(list)

    for uid, iid, true_r, est, _ in predictions:
        user_est_true[uid].append((est, true_r))

    precisions = {}
    recalls = {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        top_k = user_ratings[:k]
        relevant_items = [true_r for (_, true_r) in user_ratings if true_r >= threshold]
        recommended_items = [est for (est, true_r) in top_k if est >= threshold]

        precisions[uid] = len(recommended_items) / k if k else 0
        recalls[uid] = len(recommended_items) / len(relevant_items) if relevant_items else 0

    precision = sum(precisions.values()) / len(precisions)
    recall = sum(recalls.values()) / len(recalls)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    return precision, recall, f1


In [11]:
predictions = baseline_model.test(testset)

precision, recall, f1 = precision_recall_at_k(predictions, k=5)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


Precision: 0.2175824175824176
Recall: 1.0
F1-score: 0.3574007220216606


In [12]:
refined_model = SVD(
    n_factors=100,
    n_epochs=30,
    lr_all=0.005,
    reg_all=0.02
)

refined_model.fit(trainset)


In [13]:
refined_predictions = refined_model.test(testset)

p2, r2, f2 = precision_recall_at_k(refined_predictions, k=5)

print("Refined Precision:", p2)
print("Refined Recall:", r2)
print("Refined F1-score:", f2)


Refined Precision: 0.2175824175824176
Refined Recall: 1.0
Refined F1-score: 0.3574007220216606


In [14]:
precision_recall_at_k(refined_predictions, k=5)


(0.2175824175824176, 1.0, 0.3574007220216606)

In [15]:
precision_recall_at_k(refined_predictions, k=10)


(0.1087912087912088, 1.0, 0.19623389494549057)